阶段1：构造模型

在票价确定的前提下，预测各航段的客流量（已完成）。

阶段2：解析过程

根据客流信息分配舱位，以确定总收益。
客舱分为两类：短途类（AB或BC，称为S类）和长途类（AC，称为L类），总容量为C。

根据是否满座进行分类：

若不满座（S+L≤C），则按S、L和C的预测比例分配舱位，以实现收益最大化。
若满座（S+L>C），则需根据销售量变化预测进行取舍，并作如下判断：
若票价AB+BC>AC，则优先将舱位拆分为AB和BC段销售，S的值取min(AB, BC)，其余舱位分配给L。
若票价AB+BC<AC，则优先分配舱位给AC段，剩余舱位分配给AB和BC段。

阶段3：求总收益最大值

以票价为自变量，总收益为因变量，绘制总收益曲线，并求出总收益的最大值。

In [5]:
from sklearn.metrics import mean_squared_error

In [6]:
from tqdm import tqdm
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 加载数据并预处理
data = pd.read_csv('../../data-hh/my/202410221653-processed.csv', dtype={'aircraft': str})
categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']
label_encoders = {}
for col in categorical_columns:
    label_encoders[col] = LabelEncoder()
    data[col] = label_encoders[col].fit_transform(data[col].astype(str))
data.fillna(0, inplace=True)

# 分离特征和目标
features = ['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 
            'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 
            'second', 'from', 'to']
target = 'pax'
X = data[features]
y = data[target]

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 初始化模型
model = DecisionTreeRegressor(
    max_depth=10,               # 限制树的深度
    min_samples_leaf=5,         # 每个叶子节点最少包含 5 个样本
    min_samples_split=10,       # 分裂节点所需的最少样本数
    max_features='sqrt',        # 每次分裂时使用 √n 个特征
    max_leaf_nodes=100,         # 限制树的叶子节点数量
    min_impurity_decrease=0.01, # 分裂所需的最小纯度改善
    random_state=42             # 设置随机种子
)

# 模拟训练过程并添加进度条
batch_size = 100
trained_samples = 0
num_batches = len(X_train) // batch_size

for i in tqdm(range(0, len(X_train), batch_size), desc="Training Progress"):
    batch_X = X_train.iloc[i:i+batch_size]
    batch_y = y_train.iloc[i:i+batch_size]
    
    if i == 0:
        model.fit(batch_X, batch_y)
    else:
        model.fit(pd.concat([X_train.iloc[:i], batch_X]), pd.concat([y_train.iloc[:i], batch_y]))
    
    trained_samples += len(batch_X)

# 测试模型
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")


Training Progress: 100%|██████████████████████████████████████████████████████████████████████████████████| 7614/7614 [1:24:42<00:00,  1.50it/s]

Mean Squared Error: 1316.5983340462985


In [7]:
# 显示20条测试结果（真实值 vs 预测值）
test_results = list(zip(y_test.values[:20], y_pred[:20]))  # 真实值和预测值
print("\n20条测试结果（真实值 vs 预测值）:")
for i, (true_value, pred_value) in enumerate(test_results):
    print(f"第{i+1}条: 真实值={true_value}, 预测值={pred_value:.2f}")


20条测试结果（真实值 vs 预测值）:
第1条: 真实值=150, 预测值=126.06
第2条: 真实值=123, 预测值=82.70
第3条: 真实值=186, 预测值=126.06
第4条: 真实值=112, 预测值=82.43
第5条: 真实值=122, 预测值=122.57
第6条: 真实值=63, 预测值=81.02
第7条: 真实值=90, 预测值=150.95
第8条: 真实值=22, 预测值=89.37
第9条: 真实值=174, 预测值=155.48
第10条: 真实值=121, 预测值=118.19
第11条: 真实值=98, 预测值=109.69
第12条: 真实值=146, 预测值=126.06
第13条: 真实值=29, 预测值=94.30
第14条: 真实值=252, 预测值=204.29
第15条: 真实值=106, 预测值=126.06
第16条: 真实值=117, 预测值=126.06
第17条: 真实值=125, 预测值=118.19
第18条: 真实值=159, 预测值=122.57
第19条: 真实值=109, 预测值=82.43
第20条: 真实值=84, 预测值=109.69


In [1]:
print(666)

666
